https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients

In [ ]:
from google.colab import drive

Importing the Google Drive module from Colab to enable access to files stored in the cloud.

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


Installing the CatBoost library, which is not pre-installed in Google Colab environments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


Loading all necessary libraries:
- **pandas / numpy** for data manipulation
- **matplotlib / seaborn** for visualization
- **sklearn** for preprocessing, models, and evaluation metrics
- **XGBoost / CatBoost** for gradient boosting models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Mounting Google Drive to the Colab environment so we can read the dataset file stored there.

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Default of Credit Card Clients/default of credit card clients.xlsx')
df

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000,1,3,1,39,0,0,0,0,...,88004,31237,15980,8500,20000,5003,3047,5000,1000,0
29996,29997,150000,1,3,2,43,-1,-1,-1,-1,...,8979,5190,0,1837,3526,8998,129,0,0,0
29997,29998,30000,1,2,2,37,4,3,2,-1,...,20878,20582,19357,0,0,22000,4200,2000,3100,1
29998,29999,80000,1,3,1,41,1,-1,0,0,...,52774,11855,48944,85900,3409,1178,1926,52964,1804,1


Reading the Excel dataset into a pandas DataFrame. The dataset contains **30,000 records** and **25 columns** representing credit card client information from Taiwan.

> Add blockquote



# EDA

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

Inspecting the structure of the dataset — checking column names, data types, and confirming no null values exist across all 25 columns.

In [ ]:
df.describe()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,1.603733,1.853133,1.551867,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,0.489129,0.790349,0.521970,9.217904,1.123802,1.197186,1.196868,1.169139,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,1.000000,0.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,30000.000000,1000000.000000,2.000000,6.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


Generating descriptive statistics for all numeric features to understand the distribution, spread, and scale of each variable (e.g., mean credit limit is ~167,000 NT dollars).

In [ ]:
df.duplicated().sum()

np.int64(0)

Checking for duplicate rows. The result is **0**, meaning no duplicate entries exist in the dataset.

In [ ]:
outlier_report = []
numeric_df = df.select_dtypes(include=['int64'])
for i in numeric_df:
        Q1 = df[i].quantile(0.25)
        Q3 = df[i].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5*IQR
        upper_bound = Q3 + 1.5*IQR

        outliers_count = ((df[i] < lower_bound) | (df[i] > upper_bound)).sum()
        percentage = (outliers_count / len(df)) * 100
        outlier_report.append({
            'Column': i,
            'Outliers': outliers_count,
            'Percentage (%)': round(percentage, 2)
        })
report_df = pd.DataFrame(outlier_report)
report_df.head()

,Column,Outliers,Percentage (%)
0,ID,0,0.00
1,LIMIT_BAL,167,0.56
2,SEX,0,0.00
3,EDUCATION,454,1.51
4,MARRIAGE,0,0.00


Detecting outliers in all numeric columns using the **IQR method**. Any value below `Q1 - 1.5×IQR` or above `Q3 + 1.5×IQR` is flagged. This helps identify features with extreme values that could affect model performance.

In [ ]:
null_count = df.isnull().sum()
null_percentage = (null_count / len(df)) * 100
report = pd.DataFrame({
        'Missing Values': null_count,
        'Percentage (%)': round(null_percentage, 2)
    }).sort_values(by='Percentage (%)', ascending=False)
report.head()

,Missing Values,Percentage (%)
ID,0,0.0
LIMIT_BAL,0,0.0
SEX,0,0.0
EDUCATION,0,0.0
MARRIAGE,0,0.0


Checking for missing values across all columns. The result confirms **no missing values** in the entire dataset — no imputation is needed.

In [ ]:
cat = df[['SEX','EDUCATION','MARRIAGE','default payment next month',]]
for col in cat:
        print(f"\n Column: [{col}]")


        counts = df[col].value_counts()
        percentages = df[col].value_counts(normalize=True) * 100


        dist_df = pd.DataFrame({
            'Value': counts.index,
            'Count': counts.values,
            'Percentage (%)': percentages.values.round(4)
        })

        print(dist_df.to_string(index=False))
        print("-" * 30)


 Column: [SEX]
 Value  Count  Percentage (%)
     2  18112         60.3733
     1  11888         39.6267
------------------------------

 Column: [EDUCATION]
 Value  Count  Percentage (%)
     2  14030         46.7667
     1  10585         35.2833
     3   4917         16.3900
     5    280          0.9333
     4    123          0.4100
     6     51          0.1700
     0     14          0.0467
------------------------------

 Column: [MARRIAGE]
 Value  Count  Percentage (%)
     2  15964         53.2133
     1  13659         45.5300
     3    323          1.0767
     0     54          0.1800
------------------------------

 Column: [default payment next month]
 Value  Count  Percentage (%)
     0  23364           77.88
     1   6636           22.12
------------------------------


Analyzing the value distribution of categorical columns:
- **SEX**: 60% female, 40% male
- **EDUCATION**: majority university-educated
- **MARRIAGE**: majority single
- **Target**: ~22% defaulted, ~78% did not — confirming class imbalance

In [ ]:
for n in df.columns:
  print(n)
  print(df[n].unique())
  print("------------------------------------")

ID
[    1     2     3 ... 29998 29999 30000]
------------------------------------
LIMIT_BAL
[  20000  120000   90000   50000  500000  100000  140000  200000  260000
  630000   70000  250000  320000  360000  180000  130000  450000   60000
  230000  160000  280000   10000   40000  210000  150000  380000  310000
  400000   80000  290000  340000  300000   30000  240000  470000  480000
  350000  330000  110000  420000  170000  370000  270000  220000  190000
  510000  460000  440000  410000  490000  390000  580000  600000  620000
  610000  700000  670000  680000  430000  550000  540000 1000000  530000
  710000  560000  520000  750000  640000   16000  570000  590000  660000
  720000  327680  740000  800000  760000  690000  650000  780000  730000]
------------------------------------
SEX
[2 1]
------------------------------------
EDUCATION
[2 1 3 5 4 6 0]
------------------------------------
MARRIAGE
[1 2 3 0]
------------------------------------
AGE
[24 26 34 37 57 29 23 28 35 51 41 30 49 39 

Printing all unique values per column to verify the encoding of categorical variables and detect any undocumented or irregular values (e.g., EDUCATION has values 0, 5, 6 which are not in the official encoding).

# Data preparation

In [ ]:

df['utilization_avg'] = df[['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']].mean(axis=1) / df['LIMIT_BAL']

df['avg_payment_delay'] = df[['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']].mean(axis=1)

df['payment_ratio'] = df['PAY_AMT1'] / (df['BILL_AMT1'] + 1)

Creating 3 new engineered features to better capture client behavior:
1. **`utilization_avg`** — how much of the credit limit the client uses on average (higher = more financial pressure)
2. **`avg_payment_delay`** — average repayment delay across 6 months (higher = worse payment history)
3. **`payment_ratio`** — ratio of last payment to last bill (lower = client is not paying the full balance)

In [ ]:
df['EDUCATION'] = df['EDUCATION'].apply(lambda x: 4 if x > 4 or x == 0 else x)
df['MARRIAGE'] = df['MARRIAGE'].apply(lambda x: 3 if x == 0 else x)

Cleaning irregular category values that are not part of the official encoding:
- **EDUCATION**: undocumented values (0, 5, 6) are grouped into category 4 ('other')
- **MARRIAGE**: undocumented value (0) is merged into category 3 ('other')

In [ ]:
df = pd.get_dummies(df, columns=['EDUCATION', 'MARRIAGE'], drop_first=True)

Applying one-hot encoding to `EDUCATION` and `MARRIAGE` to convert them into numeric binary columns suitable for machine learning models. `drop_first=True` avoids the dummy variable trap (multicollinearity).

In [ ]:
dummy_cols = [col for col in df.columns if col.startswith('EDUCATION_') or col.startswith('MARRIAGE_')]
cat_df_dummies = df[dummy_cols]

for col in cat_df_dummies.columns:
        print(f"\n Column: [{col}]")

        counts = cat_df_dummies[col].value_counts()
        percentages = cat_df_dummies[col].value_counts(normalize=True) * 100

        dist_df = pd.DataFrame({
            'Value': counts.index,
            'Count': counts.values,
            'Percentage (%)': percentages.values.round(4)
        })

        print(dist_df.to_string(index=False))
        print("-" * 30)


 Column: [EDUCATION_2]
 Value  Count  Percentage (%)
 False  15970         53.2333
  True  14030         46.7667
------------------------------

 Column: [EDUCATION_3]
 Value  Count  Percentage (%)
 False  25083           83.61
  True   4917           16.39
------------------------------

 Column: [EDUCATION_4]
 Value  Count  Percentage (%)
 False  29532           98.44
  True    468            1.56
------------------------------

 Column: [MARRIAGE_2]
 Value  Count  Percentage (%)
  True  15964         53.2133
 False  14036         46.7867
------------------------------

 Column: [MARRIAGE_3]
 Value  Count  Percentage (%)
 False  29623         98.7433
  True    377          1.2567
------------------------------


Verifying the distribution of the newly created one-hot encoded columns to confirm the encoding was applied correctly.

In [ ]:
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN # Corrected import statement
import numpy as np # Import numpy for np.inf and np.nan


X = df.drop(columns=["default payment next month","ID"])
y = df["default payment next month"]

# Check for NaNs and infinities before splitting
print(f"Number of NaN values in X before cleaning: {X.isnull().sum().sum()}")
print(f"Number of infinite values in X before cleaning: {np.isinf(X).sum().sum()}")

# Fill infinite values with NaN for easier handling with fillna
X.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill NaN values with 0. You might consider median or mean depending on the feature's distribution.
# For ratio features where division by zero caused inf/NaN, 0 can be a reasonable default.
X.fillna(X.mean(), inplace=True)

print(f"Number of NaN values in X after cleaning: {X.isnull().sum().sum()}")
print(f"Number of infinite values in X after cleaning: {np.isinf(X).sum().sum()}")




X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

counts = Counter(y_train)
specific_weight = (counts[0] / counts[1])*1.5
print(f"Suggested scale_pos_weight: {specific_weight:.2f}")

Number of NaN values in X before cleaning: 8
Number of infinite values in X before cleaning: 8
Number of NaN values in X after cleaning: 0
Number of infinite values in X after cleaning: 0
Suggested scale_pos_weight: 5.28


Preparing the data for modeling:
- Separating features `X` from the target `y`
- Replacing any `inf` or `NaN` values introduced during feature engineering with column means
- Splitting into **80% training / 20% test** using stratification to preserve class distribution
- Computing `scale_pos_weight` to handle class imbalance in boosting models

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Applying `StandardScaler` to normalize features to zero mean and unit variance. This is fitted **only on training data** to prevent data leakage. Scaling is essential for Logistic Regression, KNN, and MLP which are sensitive to feature magnitude.

# result

In [ ]:
log_model = LogisticRegression(max_iter=1000,class_weight='balanced')
log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)
y_prob = log_model.predict_proba(X_test_scaled)[:,1]

print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

=== Logistic Regression ===
Accuracy: 0.6811666666666667
ROC-AUC: 0.7115557941523327
              precision    recall  f1-score   support

           0       0.87      0.70      0.77      4673
           1       0.37      0.62      0.46      1327

    accuracy                           0.68      6000
   macro avg       0.62      0.66      0.62      6000
weighted avg       0.76      0.68      0.70      6000



**Baseline Model 1 — Logistic Regression**
A simple linear model used as a baseline. `class_weight='balanced'` automatically compensates for class imbalance. Result: Accuracy 68.1%, ROC-AUC 0.712.

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model.fit(X_train_scaled, y_train)

y_pred = knn_model.predict(X_test_scaled)
y_prob = knn_model.predict_proba(X_test_scaled)[:,1]

print("=== KNN ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

=== KNN ===
Accuracy: 0.8018333333333333
ROC-AUC: 0.7133599986195933
              precision    recall  f1-score   support

           0       0.83      0.93      0.88      4673
           1       0.59      0.33      0.43      1327

    accuracy                           0.80      6000
   macro avg       0.71      0.63      0.65      6000
weighted avg       0.78      0.80      0.78      6000



**Baseline Model 2 — K-Nearest Neighbors (k=7)**
A non-parametric distance-based model. While it achieves high accuracy (80.2%), the ROC-AUC (0.713) and low recall on the default class show it struggles with class imbalance.

In [ ]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100,50),
    max_iter=1500,
    alpha = 0.01,
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    random_state=42
)

mlp_model.fit(X_train_scaled, y_train)

y_pred = mlp_model.predict(X_test_scaled)
y_prob = mlp_model.predict_proba(X_test_scaled)[:,1]

print("=== Deep Learning (MLP) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

=== Deep Learning (MLP) ===
Accuracy: 0.7881666666666667
ROC-AUC: 0.7180649923214876
              precision    recall  f1-score   support

           0       0.84      0.89      0.87      4673
           1       0.53      0.42      0.47      1327

    accuracy                           0.79      6000
   macro avg       0.69      0.66      0.67      6000
weighted avg       0.77      0.79      0.78      6000



**Baseline Model 3 — MLP Neural Network**
A feedforward neural network with two hidden layers (100, 50 neurons), ReLU activations, and an adaptive Adam optimizer. L2 regularization (`alpha=0.01`) prevents overfitting. Result: Accuracy 78.8%, ROC-AUC 0.718.

In [ ]:
xgb_model = XGBClassifier(
    scale_pos_weight=specific_weight,
    max_depth=5,
    learning_rate=0.05,
    n_estimators=200,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:,1]

print("=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

=== XGBoost ===
Accuracy: 0.6791666666666667
ROC-AUC: 0.7748363951968943
              precision    recall  f1-score   support

           0       0.89      0.67      0.76      4673
           1       0.38      0.72      0.50      1327

    accuracy                           0.68      6000
   macro avg       0.64      0.69      0.63      6000
weighted avg       0.78      0.68      0.71      6000



**Baseline Model 4 — XGBoost (initial)**
`scale_pos_weight` gives more weight to the minority (default) class to improve recall. Result: Accuracy 67.9%, ROC-AUC 0.775 — better AUC than simpler models despite lower accuracy.

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

# حساب وزن الفئة
scale_pos_weight = sum(y_train == 0) / sum(y_train == 1)*1.6

model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

param_dist = {
    'max_depth': [3,4,5,6,7],
    'learning_rate': [0.01,0.03,0.05,0.1],
    'n_estimators': [200,300,400,500],
    'subsample': [0.6,0.7,0.8,0.9,1],
    'colsample_bytree': [0.6,0.7,0.8,0.9,1],
    'min_child_weight': [1,3,5,7],
    'gamma': [0,0.1,0.3,0.5]
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=50,
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)

print("Best CV AUC:")
print(random_search.best_score_)

Fitting 10 folds for each of 50 candidates, totalling 500 fits
Best Parameters:
{'subsample': 0.7, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.01, 'gamma': 0.5, 'colsample_bytree': 0.6}
Best CV AUC:
0.786632498167601


**Hyperparameter Tuning — XGBoost with RandomizedSearchCV**
Searching over 50 random combinations of 7 hyperparameters using 10-Fold Stratified CV scored on ROC-AUC. Best CV AUC found: **0.787**. Best parameters include `max_depth=6`, `learning_rate=0.01`, `n_estimators=400`.

In [ ]:
best_model = random_search.best_estimator_
y_pred_best = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Test AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred_best))

Accuracy: 0.6601666666666667
Test AUC: 0.7801032273296016
              precision    recall  f1-score   support

           0       0.90      0.63      0.74      4673
           1       0.37      0.76      0.50      1327

    accuracy                           0.66      6000
   macro avg       0.64      0.70      0.62      6000
weighted avg       0.78      0.66      0.69      6000



Evaluating the best XGBoost model from RandomizedSearchCV on the held-out test set. Result: Accuracy 66.0%, ROC-AUC **0.780** — improved over the initial XGBoost configuration.

# best solution

In [ ]:
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

Defining the 10-Fold Stratified K-Fold cross-validator used in the final evaluation phase. Stratification ensures each fold contains approximately the same 22%/78% class ratio as the full dataset.

In [ ]:
models = {
    "XGBoost": {
        "pipeline": XGBClassifier(
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=42
        ),
        "params": {
            "max_depth": [3, 5, 7],
            "learning_rate": [0.03, 0.1],
            "n_estimators": [200, 400],
            "subsample": [0.8, 1],
            "colsample_bytree": [0.8, 1]
        }
    },

    "CatBoost": {
        "pipeline": CatBoostClassifier(
            verbose=0,
            random_state=42
        ),
        "params": {
            "max_depth": [3, 5, 7],
            "learning_rate": [0.03, 0.1],
            "n_estimators": [200, 400],
            "subsample": [0.8, 1] # Removed colsample_bytree
        }
    }
}

Defining the search space for GridSearchCV:
- **XGBoost**: tuning depth, learning rate, estimators, subsample, and column sampling
- **CatBoost**: same parameters but without `colsample_bytree` (CatBoost handles this internally)

In [ ]:
results = []

for name, config in models.items():

    grid = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=skf,
        scoring="roc_auc",   # recommended for imbalanced data
        n_jobs=-1
    )

    grid.fit(X, y)

    results.append({
        "Model": name,
        "Best Score (AUC)": grid.best_score_,
        "Best Parameters": grid.best_params_
    })

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:16:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Running GridSearchCV for both XGBoost and CatBoost, exhaustively trying all hyperparameter combinations with 10-Fold Stratified CV scored on ROC-AUC. This is the most thorough tuning phase in the notebook.

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Best Score (AUC)", ascending=False)
results_df

,Model,Best Score (AUC),Best Parameters
1,CatBoost,0.788141,"{'learning_rate': 0.03, 'max_depth': 7, 'n_est..."
0,XGBoost,0.786460,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."


Displaying the final GridSearchCV comparison table sorted by ROC-AUC:
- **CatBoost**: best AUC = **0.788**
- **XGBoost**: best AUC = **0.786**
CatBoost edges out XGBoost with `max_depth=7`, `learning_rate=0.03`, `n_estimators=400`.

In [ ]:
import ast
results_df['Best Parameters']=results_df['Best Parameters'].apply(lambda x: ast.literal_eval("{" + x + "}"))
r=results_df['Best Parameters'].apply(pd.Series)
r

,learning_rate,max_depth,n_estimators,subsample,colsample_bytree
1,0.03,7.0,400.0,1.0,NaN
0,0.03,5.0,200.0,0.8,0.8


Expanding the 'Best Parameters' dictionary column into separate columns to make it easier to compare the optimal hyperparameter values side by side across models.

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Define the model with fixed parameters
# Fixed: changed eval_metric from 'logloss' to 'Logloss'
cat_model = CatBoostClassifier(
    max_depth=7,
    learning_rate=0.03,
    n_estimators=400,
    eval_metric="Logloss",
    subsample=1,
    scale_pos_weight=3.52,
    random_state=42,
    verbose=0
)
y_pred = (y_prob > 0.3).astype(int)

# Stratified K-Fold
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Out-of-fold predictions (class labels)
y_pred = cross_val_predict(
    cat_model,
    X,
    y,
    cv=cv,
    method="predict",
    n_jobs=-1
)

# Out-of-fold predicted probabilities
y_prob = cross_val_predict(
    cat_model,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Evaluation
print("=== CatBoost with StratifiedKFold ===")
print("Accuracy:", accuracy_score(y, y_pred))
print("ROC-AUC:", roc_auc_score(y, y_prob))
print("Classification Report:")
print(classification_report(y, y_pred))

=== CatBoost with StratifiedKFold ===
Accuracy: 0.7653666666666666
ROC-AUC: 0.7865512379028792
Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.80      0.84     23364
           1       0.48      0.63      0.54      6636

    accuracy                           0.77     30000
   macro avg       0.68      0.72      0.69     30000
weighted avg       0.79      0.77      0.78     30000



**Final Model — CatBoost with 10-Fold Stratified CV**
Using the best hyperparameters from GridSearchCV. Out-of-fold predictions are generated across all 30,000 samples. A probability threshold of **0.3** (instead of default 0.5) is used to boost recall on the minority class. Final result: Accuracy 76.5%, ROC-AUC **0.787**, F1 (default class) = 0.54.

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Define the model with fixed parameters
xgb_model = XGBClassifier(
    max_depth=5,
    learning_rate=0.03,
    n_estimators=200,
    eval_metric="logloss",
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=3.52,
    random_state=42
)
y_pred = (y_prob > 0.3).astype(int)
# Stratified K-Fold
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Out-of-fold predictions (class labels)
y_pred = cross_val_predict(
    xgb_model,
    X,
    y,
    cv=cv,
    method="predict",
    n_jobs=-1
)

# Out-of-fold predicted probabilities
y_prob = cross_val_predict(
    xgb_model,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Evaluation
print("=== XGBoost with StratifiedKFold ===")
print("Accuracy:", accuracy_score(y, y_pred))
print("ROC-AUC:", roc_auc_score(y, y_prob))
print("Classification Report:")
print(classification_report(y, y_pred))

=== XGBoost with StratifiedKFold ===
Accuracy: 0.7615666666666666
ROC-AUC: 0.785718797351226
Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.80      0.84     23364
           1       0.47      0.63      0.54      6636

    accuracy                           0.76     30000
   macro avg       0.68      0.71      0.69     30000
weighted avg       0.79      0.76      0.77     30000



**Final Model — XGBoost with 10-Fold Stratified CV**
Same evaluation strategy as CatBoost using out-of-fold predictions. Final result: Accuracy 76.2%, ROC-AUC **0.786**, F1 (default class) = 0.54 — very close to CatBoost but slightly lower.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=1500,
    alpha=0.01,
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    random_state=42
)

# 10-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Out-of-fold predictions (class labels)
y_pred = cross_val_predict(
    mlp_model,
    X_train_scaled,
    y_train,
    cv=cv,
    method="predict",
    n_jobs=-1
)

# Out-of-fold predicted probabilities
y_prob = cross_val_predict(
    mlp_model,
    X_train_scaled,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Evaluation
print("=== Deep Learning (MLP) with StratifiedKFold ===")
print("Accuracy:", accuracy_score(y_train, y_pred))
print("ROC-AUC:", roc_auc_score(y_train, y_prob))
print("Classification Report:")
print(classification_report(y_train, y_pred))

=== Deep Learning (MLP) with StratifiedKFold ===
Accuracy: 0.7730416666666666
ROC-AUC: 0.7090633779714486
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.88      0.86     18691
           1       0.48      0.38      0.43      5309

    accuracy                           0.77     24000
   macro avg       0.66      0.63      0.64     24000
weighted avg       0.76      0.77      0.76     24000



**Final Model — MLP Neural Network with 10-Fold Stratified CV**
Applied on the training set (X_train_scaled, y_train) using out-of-fold predictions. Final result: Accuracy 77.3%, ROC-AUC **0.709**, F1 (default class) = 0.43 — lower AUC than the boosting models, indicating gradient boosting is better suited for this tabular dataset.